# Step 2 -- Feature Selection

Narrow a preprocessed dataset down to the features worth sending into causal discovery.

Pipeline:
> **Variance filter -> Correlation drop -> Random Forest importance -> Final set**

**Input**: `step1_cleaned.csv` (output from notebook 01)  
**Output**: `step2_selected.csv`

**Only change the `CONFIG` cell.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# -- CHANGE THESE ---------------------------------------------------------------
FILE_PATH     = r""                    # output CSV from notebook 01
TS_COL        = "date_time"            # timestamp column (or None)

TARGET_COL    = None    # if you already know the target column -- Phase 1 is skipped
                        # e.g. 'wtrm_avg_TrmTmp_GbxBrg152'
ALARM_PATTERN = "alarm_"  # substring that identifies fault/alarm columns
                          # used in Phase 1 to find the best target automatically
                          # set None if your data has no alarm columns

# Thresholds
VAR_THRESH    = 0.01    # drop columns with variance < this fraction of mean variance
CORR_THRESH   = 0.92    # drop one of each pair with |r| > this
TOP_N_RF      = 15      # keep at most this many features after Phase 2

OUTPUT_PATH   = "step2_selected.csv"
# -------------------------------------------------------------------------------

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})
sns.set_style('whitegrid')
print('Config OK')


## 1 -- Load

In [ ]:
df = pd.read_csv(FILE_PATH)

if TS_COL and TS_COL in df.columns:
    df[TS_COL] = pd.to_datetime(df[TS_COL], errors='coerce')
    df = df.sort_values(TS_COL).reset_index(drop=True)

features = [c for c in df.columns
            if c not in (TS_COL, TARGET_COL) and pd.api.types.is_numeric_dtype(df[c])]

print(f'Rows     : {len(df):,}')
print(f'Features : {len(features)}')
df[features].describe().round(3)

## 2 -- Variance Filter

Drop near-constant columns. A column that barely changes carries almost no information.

The threshold is relative: `variance < VAR_THRESH x mean_variance_of_all_columns`.

In [ ]:
var_series = df[features].var().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(max(8, len(features) * 0.5), 4))
colors = ['tomato' if v < VAR_THRESH * var_series.mean() else 'steelblue'
          for v in var_series]
ax.bar(range(len(var_series)), var_series.values, color=colors)
ax.axhline(VAR_THRESH * var_series.mean(), color='red', linestyle='--',
           linewidth=1, label=f'Drop threshold  (VAR_THRESH={VAR_THRESH})')
ax.set_xticks(range(len(var_series)))
ax.set_xticklabels(var_series.index, rotation=90, fontsize=7)
ax.set_ylabel('Variance')
ax.set_title('Feature Variance  (red bars will be dropped)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
thresh_abs   = VAR_THRESH * var_series.mean()
dropped_var  = var_series[var_series < thresh_abs].index.tolist()
kept_var     = [c for c in features if c not in dropped_var]

print(f'Dropped (low variance) : {len(dropped_var)}  -- {dropped_var}')
print(f'Remaining              : {len(kept_var)}')

## 3 -- Correlation Heatmap & Redundancy Drop

When two features have |r| > `CORR_THRESH`, one is redundant -- we keep the one with *higher* mean absolute correlation to all other features (i.e. the more "connected" one, which tends to be more informative).

The heatmap shows all remaining features after variance filtering.

In [ ]:
corr = df[kept_var].corr()

sz  = max(6, len(kept_var) * 0.7)
fig, ax = plt.subplots(figsize=(sz, sz * 0.85))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)   # upper triangle
sns.heatmap(
    corr, ax=ax, mask=~mask,
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    annot=(len(kept_var) <= 15), fmt='.2f',
    linewidths=0.3, square=True,
)
ax.set_title(f'Correlation  |r| > {CORR_THRESH} = redundant pair', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Greedy drop: scan upper triangle, drop the column with lower mean |corr|
corr_abs = corr.abs()
dropped_corr = set()
for i in range(len(kept_var)):
    for j in range(i + 1, len(kept_var)):
        a, b = kept_var[i], kept_var[j]
        if a in dropped_corr or b in dropped_corr:
            continue
        if corr_abs.loc[a, b] > CORR_THRESH:
            # drop the one with lower mean abs correlation (less connected)
            ma = corr_abs[a].drop(index=a).mean()
            mb = corr_abs[b].drop(index=b).mean()
            dropped_corr.add(b if ma >= mb else a)

kept_corr = [c for c in kept_var if c not in dropped_corr]
print(f'Dropped (high correlation): {len(dropped_corr)}  -- {sorted(dropped_corr)}')
print(f'Remaining                 : {len(kept_corr)}')

## 4 -- Two-Phase Random Forest

**Phase 1** -- *Which sensor is most correlated with faults?*  
RF is trained with all surviving sensors as inputs and `alarm_active` (any alarm on/off) as output.  
The sensor with the highest OOB importance becomes the **target (Y)**.  
Skipped automatically if `TARGET_COL` is already set in CONFIG.

**Phase 2** -- *Which sensors best predict that target?*  
RF is trained with all other sensors as inputs and the Phase 1 target as output.  
Top `TOP_N_RF` sensors by importance become the **selected features (X)**.

| CONFIG | Behaviour |
|---|---|
| `TARGET_COL = 'col'` | Phase 1 skipped -- go straight to Phase 2 |
| `TARGET_COL = None` + `ALARM_PATTERN` set | Phase 1 runs, finds target from alarm correlation |
| Both None | No fault signal -- falls back to first feature as proxy |

In [ ]:
# ---- Phase 1: find best response variable --------------------------------
# Skipped if TARGET_COL is already set in CONFIG.

_target = TARGET_COL   # local copy so we don't mutate the config variable

if _target is not None:
    print(f'Phase 1 skipped -- TARGET_COL already set: "{_target}"')
else:
    alarm_cols = []
    if ALARM_PATTERN:
        alarm_cols = [c for c in df.columns if ALARM_PATTERN.lower() in c.lower()]

    if alarm_cols:
        print(f'Phase 1 -- finding best sensor correlated with fault signal')
        print(f'  Alarm/fault columns ({len(alarm_cols)}): {alarm_cols}')

        X_p1 = df[kept_corr].dropna()
        alarm_active = (df.loc[X_p1.index, alarm_cols]
                        .apply(pd.to_numeric, errors='coerce')
                        .sum(axis=1) > 0).astype(int)
        print(f'  Fault-active rows: {alarm_active.sum():,} / {len(alarm_active):,}'
              f'  ({100*alarm_active.mean():.1f}%)')

        rf1 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        rf1.fit(X_p1.values, alarm_active.values)
        imp1 = pd.Series(rf1.feature_importances_, index=kept_corr).sort_values(ascending=False)

        _target = imp1.index[0]
        print(f'\nPhase 1 result -- best response variable: "{_target}"')
        print(f'Top 8 sensors by alarm correlation:')
        print(imp1.head(8).round(4).to_string())

        # bar chart for Phase 1
        fig, ax = plt.subplots(figsize=(10, max(3, len(imp1) * 0.25)))
        ax.barh(imp1.index[::-1], imp1.values[::-1], color='steelblue')
        ax.axvline(imp1.iloc[0], color='red', linestyle='--', linewidth=1,
                   label=f'winner: {_target}')
        ax.set_xlabel('Importance (sensors -> fault signal)')
        ax.set_title('Phase 1 -- sensor correlation with fault/alarm')
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

    else:
        _target = kept_corr[0]
        print('Phase 1 skipped -- no alarm columns found (ALARM_PATTERN matched nothing).')
        print(f'  Falling back to proxy target: "{_target}" (first surviving feature)')
        print('  Tip: set TARGET_COL directly if you know your target column.')

print(f'\nTarget for Phase 2: "{_target}"')


In [ ]:
# ---- Phase 2: find best predictors of the target -------------------------
print(f'Phase 2 -- finding best predictors of "{_target}"')

if _target not in df.columns:
    print(f'ERROR: target "{_target}" not found in dataframe.')
    imp = pd.Series(dtype=float)
else:
    input_cols = [c for c in kept_corr if c != _target]
    X_p2 = df[input_cols].dropna()
    y_p2 = df.loc[X_p2.index, _target]
    common = X_p2.index.intersection(y_p2.dropna().index)
    X_p2, y_p2 = X_p2.loc[common], y_p2.loc[common]

    rf2 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf2.fit(X_p2.values, y_p2.values)
    imp = pd.Series(rf2.feature_importances_, index=input_cols).sort_values(ascending=False)

    print(f'  RF fitted on {len(X_p2):,} rows, {len(input_cols)} input features')
    print(f'  Top 10 predictors:')
    print(imp.head(10).round(4).to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(10, max(4, len(imp) * 0.38)))
colors = ['darkorange' if i < TOP_N_RF else 'steelblue' for i in range(len(imp))]
ax.barh(imp.index[::-1], imp.values[::-1], color=colors[::-1])
ax.axvline(imp.iloc[min(TOP_N_RF, len(imp)) - 1],
           color='red', linestyle='--', linewidth=1,
           label=f'Top-{TOP_N_RF} cut-off')
ax.set_xlabel('Importance (mean decrease in impurity)')
ax.set_title(f'Phase 2 RF Importance -- target: [{_target}]')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
top_features = imp.head(TOP_N_RF).index.tolist()

# include the target column itself so it is saved with the features
if _target not in top_features:
    top_features.append(_target)

print(f'Selected {len(top_features)} features (including target):')
for feat in top_features:
    score = imp[feat] if feat in imp else float('nan')
    marker = '  <- target' if feat == _target else ''
    print(f'  {feat:<45}  importance = {score:.4f}{marker}')


## 5 -- Pairwise Scatter (selected features)

Scatter plots coloured by **time** (early = blue, late = orange).  
A rotating cloud = stationarity. A drifting streak = non-stationarity. L- or U-shapes = non-linear relations that LiNGAM will pick up.

In [ ]:
plot_features = top_features[:8]   # cap at 8 to keep the plot manageable
plot_df = df[plot_features].dropna().reset_index(drop=True)
n_pts   = len(plot_df)
color   = plt.cm.plasma(np.linspace(0.1, 0.9, n_pts))

nf = len(plot_features)
fig, axs = plt.subplots(nf, nf, figsize=(2.8 * nf, 2.8 * nf))

for i, ci in enumerate(plot_features):
    for j, cj in enumerate(plot_features):
        ax = axs[i][j]
        if i == j:
            ax.hist(plot_df[ci], bins=30, color='steelblue', alpha=0.7)
            ax.set_title(ci[:20], fontsize=7)
        else:
            ax.scatter(plot_df[cj], plot_df[ci], c=color, s=2, alpha=0.5, rasterized=True)
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle('Pairwise scatter (coloured by time: blue=early -> orange=late)', fontsize=10, y=1.002)
plt.tight_layout()
plt.show()

## 6 -- Time Series of Selected Features

Final stacked ECG view -- what causal discovery will see.

In [ ]:
x = df[TS_COL] if (TS_COL and TS_COL in df.columns) else df.index
nf = len(top_features)
fig, axs = plt.subplots(nf, 1, figsize=(15, 2.5 * nf), sharex=True)
if nf == 1: axs = [axs]

colors_cycle = plt.cm.tab10.colors
for k, (ax, col) in enumerate(zip(axs, top_features)):
    ax.plot(x, df[col], linewidth=0.8, color=colors_cycle[k % 10])
    ax.set_ylabel(col[:30], fontsize=8, labelpad=4)

if TS_COL and TS_COL in df.columns:
    axs[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    axs[-1].tick_params(axis='x', rotation=30)

fig.suptitle('Selected features -- time series', fontsize=12, y=1.002)
fig.tight_layout()
plt.show()

## 7 -- Save

In [ ]:
save_cols = ([TS_COL] if (TS_COL and TS_COL in df.columns) else []) + top_features
df[save_cols].to_csv(OUTPUT_PATH, index=False)
print(f'Saved  ->  {OUTPUT_PATH}')
print(f'Shape  :  {df[save_cols].shape}')
print(f'Columns: {save_cols}')